**Дистилляция 2.0: ансамбль → одна модель, с псевдо-лейблами архива** (база: distillation_final v7)

Две фазы одного ноутбука (`PHASE` в пульте):

1. **PHASE=predict** (~2-3 ч): preprocessed-обработка архивных CT (nii.gz) тем же
   конвейером nnUNet (`run_case_npy`: transpose → crop → normalize → resample) →
   sliding-window предсказание 5-фолдового учителя (fp16, без TTA — та же конвенция,
   что при обучении) → **псевдо-маска** (argmax среднего softmax) + **уверенность**
   (средний max-prob по форграунду) + `class_locations` для оверсэмплинга →
   `.b2nd + _seg.b2nd + .pkl + _conf.json` в `output/archive_out/`.
2. Вручную: скачать output → залить как приватный датасет `distill-archive-pseudo`
   (заливка через API с KGAT отдаёт 500 — только через веб-интерфейс).
3. **PHASE=train** (resume-цепочка): GT-сплит 85/15 как в v7 (seed 20261003) +
   топ-N архивных кейсов по уверенности в train (val — только GT) → дистилляция
   (учитель на лету, soft CE + soft Dice + hard-GT вес) → NNCF INT8 → валидация.

**Прикреплённые Input'ы:**
- predict: `valiadtion-nifti` (учителя + preprocessed), `distill-archive` (nii.gz тома);
- train: `valiadtion-nifti`, `distill-archive-pseudo` (output фазы predict),
  с сессии 2+: `nnunet-archive-ckpt` (страховочные чекпоинты этой цепочки).

**Параметры выбраны 2026-11-03:** HARD_GT=0.2 · INIT=fold_1 (лучший фолд) ·
N_ARCHIVE=100 · EPOCHS=150 · LR=0.01 · DA5=0 · OVERSAMPLE=0.33.

**Гейт новой модели (локально, 10 внешних кейсов):** mean Dice ≥ 0.8394 (не хуже
текущего прода distilled_int8) и малые структуры не хуже −0.03. Kaggle-валидация
(ячейка в конце) — контаминированная шкала, только для контроля конвейера.

Заметка о hard-GT на архивных кейсах: hard-компонента лосса действует на ВСЕ
батчи; на GT-кейсах это коррекция ошибок учителя, на архивных — уверенный
self-training по отфильтрованным псевдо-маскам (классическая схема self-training).


In [ ]:
# ЯЧЕЙКА 1 — ПУЛЬТ УПРАВЛЕНИЯ (все экспериментальные параметры)
# ⚙ = можно/нужно крутить. Меняйте ОДИН параметр за прогон.
# ============================================================
import os as _os

# ---------- фаза и архив ----------
_os.environ.setdefault("PHASE", "predict")            # ⚙ 'predict' → 'train' (после заливки датасета)
_os.environ.setdefault("ARCHIVE_CONF_MIN", "0.6")     # ⚙ фильтр уверенности ансамбля по форграунду
_os.environ.setdefault("N_ARCHIVE", "100")            # ⚙ топ-N архивных кейсов по уверенности

# ---------- инфраструктура (не трогать без нужды) ----------
_os.environ.setdefault("DISTILL_BUDGET_H", "11.3")    # мягкий бюджет сессии, ч
_os.environ.setdefault("DISTILL_REQUIRE_CKPT", "0")   # сессия 1 архива; с сессии 2 ставьте 1
# _os.environ["KAGGLE_BEARER"] = "KGAT_..."           # ⚠️ НЕ вшивать! Только Kaggle Secrets (Add-ons)

# ---------- экспериментальные (выбор 2026-11-03) ----------
_os.environ.setdefault("DISTILL_EPOCHS", "150")       # ⚙ длина poly-цикла. НЕ менять при resume
_os.environ.setdefault("DISTILL_LR", "0.01")          # ⚙ peak LR (fold-старт). Для INIT=distill: 0.005-0.007
_os.environ.setdefault("DISTILL_W_DICE", "0.5")       # ⚙ баланс soft Dice : soft CE
_os.environ.setdefault("DISTILL_HARD_GT", "0.2")      # ⚙ вес hard-GT (выбрано 2026-11-03)
_os.environ.setdefault("DISTILL_USE_DA5", "0")        # ⚙ сильная аугментация DA5
_os.environ.setdefault("DISTILL_OVERSAMPLE", "0.33")  # ⚙ доля форграунд-патчей
_os.environ.setdefault("DISTILL_INIT", "fold1")       # ⚙ fold0|fold1|...|distill|none (выбран fold_1)
_os.environ.setdefault("DISTILL_SMOKE", "0")          # смоук: predict — 3 кейса, train — 1 эп × 30 итер
_os.environ.setdefault("DISTILL_CPU", "0")            # CPU_MODE (только смоук!)
_os.environ.setdefault("UPLOAD_EVERY", "5")           # аплоад чекпоинта каждые N эпох

# Пульт управления: все переменные

Меняйте **ОДИН параметр за прогон** (иначе не поймёте причину изменения результата).

| Переменная | Что это | Дефолт (выбор 03.11) | Больше → | Меньше → |
|---|---|---|---|---|
| `PHASE` | Фаза: 'predict' (учитель → псевдо-маски) или 'train' (дистилляция) | predict | — | — |
| `ARCHIVE_CONF_MIN` | Порог уверенности ансамбля (средний max-prob по форграунду) для кейса архива | 0.6 | меньше кейсов, чище псевдо-метки | больше кейсов, шумнее метки |
| `N_ARCHIVE` | Топ-N архивных кейсов по уверенности в train | 100 | архив доминирует (максимум тёмных знаний, выше доля ошибок учителя) | консервативнее (GT преобладает) |
| `DISTILL_EPOCHS` | Длина цикла и poly-LR | 150 (выбрано) | дольше, выигрыш не доказан | риск недобора после рестарта |
| `DISTILL_LR` | Peak LR SGD+PolyLR | 0.01 (fold-старт) | — | для INIT=distill ставьте 0.005-0.007 |
| `DISTILL_W_DICE` | Баланс softDice:softCE | 0.5 | →1.0 объёмы/крупные | →0.0 границы/мелкие |
| `DISTILL_HARD_GT` | Вес hard-GT: итог = (1−w)·soft + w·hard | **0.2 (выбрано)** | 0.3 — сильнее якорь GT, быстрее теряются тёмные знания, на архиве сильнее self-training | 0.0 — чистая дистилляция, без коррекции малых |
| `DISTILL_USE_DA5` | Сильная аугментация DA5 | 0 | 1: +10-15% времени, кандидат при переобучении | — |
| `DISTILL_OVERSAMPLE` | Доля форграунд-патчей | 0.33 | 0.4-0.5: больше мелких структур, риск дисбаланса | — |
| `DISTILL_INIT` | Старт весов: fold0/fold1/distill/none | **fold1 (выбрано, лучший фолд)** | — | none — холодный старт (не сойдётся за 150) |
| `DISTILL_BUDGET_H` | Мягкий бюджет сессии | 11.3 (не трогать) | — | — |
| `DISTILL_REQUIRE_CKPT` | Запрет старта без чекпоинта продолжения | 0 (сессия 1) | с сессии 2 ставьте 1 (не жечь квоту впустую) | — |
| `DISTILL_SMOKE` / `DISTILL_CPU` | Смоук / CPU-прогон | 0 / 0 | — | — |
| `UPLOAD_EVERY` | Чекпоинт в датасет каждые N эпох | 5 | окно потери = N·12.5 мин | — |

Зафиксировано в коде: веса deep supervision `1/2ⁱ`; seed GT-сплита `20261003`
(тот же, что в v7 — сплит идентичен, сравнимость); batch 2, patch 56×192×192 —
из plans учителя; учитель без TTA и в predict, и в train (консистентность сигнала);
калибровка NNCF — на патчах GT-val (как в v7).

**Порядок запуска:** v1 `PHASE=predict` (SMOKE=1 сначала!) → скачать output →
залить `distill-archive-pseudo` → v2 `PHASE=train` → resume-сессии до эпохи 150 →
квантизация+валидация в конце последней сессии.

In [ ]:
# ЯЧЕЙКА 2 — установка, импорты, производные флаги
# ============================================================
import subprocess, sys, os, json
try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

def _pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

try:
    import nnunetv2  # noqa: F401
    print(f"[setup] nnunetv2 {nnunetv2.__version__}")
except ImportError:
    print("[setup] ставлю nnunetv2==2.8.1 ...")
    _pip("nnunetv2==2.8.1")

import numpy as np  # noqa: E402
import torch  # noqa: E402

PHASE = os.environ.get("PHASE", "predict").strip().lower()
RUN_TRAIN = (PHASE == "train")
SMOKE = os.environ.get("DISTILL_SMOKE") == "1"
CPU_MODE = os.environ.get("DISTILL_CPU") == "1"
NUM_EPOCHS = int(os.environ["DISTILL_EPOCHS"])
PEAK_LR = float(os.environ["DISTILL_LR"])
W_DICE = float(os.environ["DISTILL_W_DICE"])
HARD_GT_WEIGHT = float(os.environ["DISTILL_HARD_GT"])
USE_DA5 = os.environ.get("DISTILL_USE_DA5") == "1"
OVERSAMPLE = float(os.environ["DISTILL_OVERSAMPLE"])
BUDGET_H = float(os.environ["DISTILL_BUDGET_H"])
UPLOAD_EVERY = int(os.environ["UPLOAD_EVERY"])
ARCHIVE_CONF_MIN = float(os.environ["ARCHIVE_CONF_MIN"])
N_ARCHIVE = int(os.environ["N_ARCHIVE"])
INIT_SOURCE = os.environ.get("DISTILL_INIT", "fold1").strip().lower()
REQUIRE_CKPT = (os.environ.get("DISTILL_REQUIRE_CKPT") == "1") and not SMOKE and not CPU_MODE

print(f"[cfg] PHASE={PHASE} epochs={NUM_EPOCHS} lr={PEAK_LR} w_dice={W_DICE} hard_gt={HARD_GT_WEIGHT} "
      f"init={INIT_SOURCE} n_archive={N_ARCHIVE} conf_min={ARCHIVE_CONF_MIN} "
      f"da5={USE_DA5} oversample={OVERSAMPLE} budget={BUDGET_H}h smoke={SMOKE} cpu={CPU_MODE}")

In [ ]:
# ЯЧЕЙКА 3 — поиск входов (учитель + preprocessed)
# Фильтр учителя: у кандидата должны лежать ВСЕ 5 checkpoint_final.
# ============================================================
import glob

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

def _is_teacher_results(d):
    return all(os.path.isfile(os.path.join(d, f"fold_{i}", "checkpoint_final.pth"))
               for i in range(5))

in_pp = sorted(glob.glob("/kaggle/input/**/nnUNet_preprocessed/Dataset*", recursive=True))
IN_PREPROCESSED = next((h for h in in_pp if glob.glob(os.path.join(h, "*", "*.b2nd"))),
                       in_pp[0] if in_pp else None)
in_res = sorted(glob.glob("/kaggle/input/**/nnUNet_results/Dataset*/*__*__3d_fullres", recursive=True))
IN_RESULTS = next((h for h in in_res if _is_teacher_results(h)), None)
assert IN_PREPROCESSED, "nnUNet_preprocessed не найден в /kaggle/input (приаттачьте valiadtion-nifti)"
assert IN_RESULTS, "nnUNet_results учителя не найден в /kaggle/input (приаттачьте valiadtion-nifti)"
print(f"[setup] preprocessed: {IN_PREPROCESSED}")
print(f"[setup] teacher results: {IN_RESULTS}")

In [ ]:
# ЯЧЕЙКА 4 — preprocessed в /kaggle/tmp, переменные окружения
# PP в /kaggle/tmp (НЕ в working): output остаётся лёгким (архив/чекпоинты).
# ============================================================
import shutil, time

PP = "/kaggle/tmp/nnUNet_preprocessed"
DATASET_DIR = os.path.join(PP, os.path.basename(IN_PREPROCESSED.rstrip("/")))
if not os.path.isdir(DATASET_DIR):
    print("[setup] копирую preprocessed в tmp ...")
    t0 = time.time()
    shutil.copytree(IN_PREPROCESSED, DATASET_DIR)
    print(f"[setup] скопировано за {time.time() - t0:.0f} с")

os.environ["nnUNet_preprocessed"] = PP
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw"
os.environ.setdefault("nnUNet_n_proc_DA", "4")
os.makedirs(os.environ["nnUNet_results"], exist_ok=True)
os.makedirs(os.environ["nnUNet_raw"], exist_ok=True)

In [ ]:
# ЯЧЕЙКА 5 — планы, dataset.json, фабрика сети
# ============================================================
from batchgenerators.utilities.file_and_folder_operations import join, load_json  # noqa: E402
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer  # noqa: E402
from nnunetv2.training.nnUNetTrainer.variants.data_augmentation.nnUNetTrainerNoMirroring import (  # noqa: E402
    nnUNetTrainerNoMirroring)
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager  # noqa: E402

PLANS_FILE = join(DATASET_DIR, "nnUNetResEncUNetMPlans.json")
if not os.path.isfile(PLANS_FILE):
    cand = glob.glob(join(DATASET_DIR, "*Plans*.json"))
    assert cand, f"plans не найден в {DATASET_DIR}"
    PLANS_FILE = cand[0]
plans = load_json(PLANS_FILE)

# страховка: CTNormalization берёт клип-значения из plans
# (foreground_intensity_properties_per_channel) — если ключа нет, восстанавливаем
# из dataset_fingerprint.json
if "foreground_intensity_properties_per_channel" not in plans:
    fp_path = join(DATASET_DIR, "dataset_fingerprint.json")
    assert os.path.isfile(fp_path), "в plans нет foreground_intensity_properties и нет dataset_fingerprint.json"
    fp = load_json(fp_path)
    plans["foreground_intensity_properties_per_channel"] = {
        "0": {"mean": fp["mean"], "std": fp["std"],
              "min": float(fp["percentile_00_5"]), "max": float(fp["percentile_99_5"]),
              "percentile_00_5": float(fp["percentile_00_5"]),
              "percentile_99_5": float(fp["percentile_99_5"])}}
    print("[setup] foreground_intensity_properties восстановлены из fingerprint")

DATASET_JSON = join(DATASET_DIR, "dataset.json")
if not os.path.isfile(DATASET_JSON):
    organ_labels = ["BrachialPlexus_R", "BrachialPlexus_L", "SpinalCanal", "SpinalCord",
                    "Trachea", "Esophagus", "Thyroid", "A_Carotid_R", "A_Carotid_L",
                    "Larynx", "PharynxConst_I", "PharynxConst_M", "PharynxConst_S",
                    "OpticNerve_R", "OpticNerve_L", "Brain", "Submandibular_R",
                    "Submandibular_L", "Parotid_R", "Parotid_L", "Lips", "Tongue",
                    "Mandible", "Eye_R", "Lens_R", "Eye_L", "Lens_L", "TMJoint_R",
                    "TMJoint_L", "Cochlea_R", "Cochlea_L"]
    labels = {"background": 0}
    labels.update({n: i + 1 for i, n in enumerate(organ_labels)})
    dj = {"channel_names": {"0": "ct"}, "labels": labels,
          "numTraining": 100, "file_ending": ".nii.gz"}
    with open(DATASET_JSON, "w") as f:
        json.dump(dj, f, indent=2)
    print("[setup] dataset.json восстановлен")
dataset_json = load_json(DATASET_JSON)

CONFIG = "3d_fullres"
pm = PlansManager(plans)
cm = pm.get_configuration(CONFIG)

def build_net():
    from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels
    label_manager = pm.get_label_manager(dataset_json)
    num_in = determine_num_input_channels(pm, cm, dataset_json)
    try:
        # nnunetv2 >= 2.7: принимает plans/configuration managers напрямую
        return nnUNetTrainer.build_network_architecture(
            pm, cm, num_in, label_manager.num_segmentation_heads, True)
    except TypeError:
        # nnunetv2 2.6.x: имя класса архитектуры + kwargs из plans
        return nnUNetTrainer.build_network_architecture(
            cm.network_arch_class_name, cm.network_arch_init_kwargs,
            cm.network_arch_init_kwargs_req_import,
            num_in, label_manager.num_segmentation_heads, True)

In [ ]:
# ЯЧЕЙКА 6 — загрузка 5 учителей (fp16 на cuda:1; fp32/CPU в CPU_MODE)
# ============================================================
print("[teachers] загрузка 5 фолдов учителя...")
t0 = time.time()
teacher_nets = []
teacher_device = "cpu" if CPU_MODE else ("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
print(f"[teachers] устройство учителей: {teacher_device}")
for fold in range(5):
    sd = torch.load(join(IN_RESULTS, f"fold_{fold}", "checkpoint_final.pth"),
                    map_location="cpu", weights_only=False)
    net = build_net()
    net.load_state_dict(sd["network_weights"])
    net.eval()
    net.to(teacher_device)
    if not CPU_MODE:
        net.half()
    for p in net.parameters():
        p.requires_grad_(False)
    teacher_nets.append(net)
    del sd
    print(f"[teachers] fold {fold}: ok ({time.time() - t0:.0f} с)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ЯЧЕЙКА 7 — [PHASE=predict] preprocessing архива + учитель → псевдо-маски
# Тот же конвейер, что у DefaultPreprocessor (run_case_npy: transpose → crop
# по ненулевым CT → CTNormalization (клип из plans) → ресемплинг plans-функцией).
# Сегментация не нужна: учителя дают псевдо-маску. class_locations для
# оверсэмплинга пересчитываются от ПСЕВДО-маски. SMOKE=1 → первые 3 кейса.
# ============================================================
if RUN_TRAIN:
    print("[predict] PHASE=train — ячейка пропущена")
else:
    import re
    from nnunetv2.preprocessing.preprocessors.default_preprocessor import DefaultPreprocessor
    from nnunetv2.training.dataloading.nnunet_dataset import nnUNetDatasetBlosc2, comp_blosc2_params

    cands = sorted(d for d in glob.glob("/kaggle/input/*")
                   if "archive" in d.lower() and "pseudo" not in d.lower())
    assert cands, "Не найден датасет с архивными nii.gz — создайте distill-archive и приаттачьте"
    ARCHIVE_RAW = cands[0]
    raw_files = sorted(glob.glob(join(ARCHIVE_RAW, "**", "*.nii.gz"), recursive=True))
    assert raw_files, f"В {ARCHIVE_RAW} нет .nii.gz"
    OUT_DIR = "/kaggle/working/archive_out"
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"[predict] датасет архива: {ARCHIVE_RAW}, кейсов: {len(raw_files)}")

    PZ, PY, PX = tuple(cm.patch_size)

    def _steps(d, p):
        if d <= p:
            return [0]
        s = max(1, int(p * 0.5))
        st = list(range(0, d - p + 1, s))
        if st[-1] + p < d:
            st.append(d - p)
        return st

    @torch.no_grad()
    def teacher_predict_volume(vol):
        # sliding-window, stride 0.5, батч патчей; средний softmax 5 фолдов fp16.
        # без TTA — та же конвенция, что у учителя при обучении (консистентность)
        origins = [(z, y, x) for z in _steps(vol.shape[0], PZ)
                   for y in _steps(vol.shape[1], PY) for x in _steps(vol.shape[2], PX)]
        acc = np.zeros((32,) + vol.shape, dtype=np.float32)
        wsum = np.zeros(vol.shape, dtype=np.float32)
        B = 4
        for i in range(0, len(origins), B):
            chunk = origins[i:i + B]
            patches = np.stack([vol[z:z + PZ, y:y + PY, x:x + PX] for z, y, x in chunk])[:, None]
            t = torch.from_numpy(patches).to(teacher_device)
            if not CPU_MODE:
                t = t.half()
            accb = None
            for net in teacher_nets:
                out = net(t)
                if isinstance(out, (list, tuple)):
                    out = out[0]
                p = torch.softmax(out.float(), dim=1)
                accb = p if accb is None else accb + p
            accb = (accb / len(teacher_nets)).cpu().numpy()
            for j, (z, y, x) in enumerate(chunk):
                acc[:, z:z + PZ, y:y + PY, x:x + PX] += accb[j]
                wsum[z:z + PZ, y:y + PY, x:x + PX] += 1
        acc /= np.maximum(wsum, 1e-8)[None]
        return acc

    pre = DefaultPreprocessor(verbose=False)
    rw = pm.image_reader_writer_class()()
    label_manager = pm.get_label_manager(dataset_json)
    fg_labels = [int(l) for l in label_manager.foreground_labels]
    print(f"[predict] форграунд-классов: {len(fg_labels)}")

    files = raw_files[:3] if SMOKE else raw_files
    seen_ids = set()
    for path in files:
        base = os.path.basename(path).replace(".nii.gz", "")
        case_id = "arch_" + re.sub(r"[^0-9A-Za-z_]+", "_", base)[:60]
        while case_id in seen_ids:
            case_id += "_x"
        seen_ids.add(case_id)
        trunc = join(OUT_DIR, case_id)
        if os.path.isfile(trunc + ".b2nd"):
            print(f"[predict] {case_id}: уже готов — пропуск")
            continue
        t0 = time.time()
        data, props = rw.read_images([path])
        data2, _gen, props2 = pre.run_case_npy(data, None, props, pm, cm, dataset_json)
        vol = data2[0]
        pads = (max(0, PZ - vol.shape[0]), max(0, PY - vol.shape[1]), max(0, PX - vol.shape[2]))
        if any(pads):
            vol = np.pad(vol, [(0, pads[0]), (0, pads[1]), (0, pads[2])])
            data2 = np.pad(data2, [(0, 0), (0, pads[0]), (0, pads[1]), (0, pads[2])])
        probs = teacher_predict_volume(vol.astype(np.float32))
        seg = probs.argmax(0).astype(np.int16)[None]          # (1,Z,Y,X)
        fgmask = seg[0] > 0
        nfg = int(fgmask.sum())
        conf_fg = float(probs.max(0)[fgmask].mean()) if nfg > 0 else 0.0
        fg_fraction = float(fgmask.mean())
        props2["class_locations"] = pre._sample_foreground_locations(seg, fg_labels, seed=1234)
        bd, cd = comp_blosc2_params(data2.shape, tuple(cm.patch_size), data2.itemsize)
        bs, cs = comp_blosc2_params(seg.shape, tuple(cm.patch_size), seg.itemsize)
        nnUNetDatasetBlosc2.save_case(data2.astype(np.float32), seg.astype(np.int16),
                                      props2, trunc, chunks=cd, blocks=bd,
                                      chunks_seg=cs, blocks_seg=bs)
        with open(join(OUT_DIR, case_id + "_conf.json"), "w") as f:
            json.dump({"case": case_id, "conf_fg": conf_fg, "fg_fraction": fg_fraction,
                       "n_fg": nfg, "shape": list(int(s) for s in vol.shape)}, f)
        print(f"[predict] {case_id}: shape {vol.shape}, conf_fg {conf_fg:.3f}, "
              f"fg {fg_fraction * 100:.1f}% ({time.time() - t0:.0f} с)")
    print(f"[predict] обработано кейсов: {len(files)}")

In [ ]:
# ЯЧЕЙКА 8 — [PHASE=predict] фильтр уверенности + манифест
# kept = conf_fg >= ARCHIVE_CONF_MIN и правдоподобная доля форграунда.
# N_ARCHIVE НЕ режется здесь — выбор топ-N в train-фазе (по манифесту).
# ============================================================
if RUN_TRAIN:
    print("[predict] PHASE=train — ячейка пропущена")
else:
    OUT_DIR = "/kaggle/working/archive_out"
    recs = []
    for p in sorted(glob.glob(join(OUT_DIR, "*_conf.json"))):
        with open(p) as f:
            recs.append(json.load(f))
    assert recs, "Нет метрик кейсов — predict-ячейка не выполнена?"
    for r in recs:
        r["kept"] = bool(r["conf_fg"] >= ARCHIVE_CONF_MIN and 0.005 <= r["fg_fraction"] <= 0.5)
    kept = sorted((r for r in recs if r["kept"]), key=lambda r: -r["conf_fg"])
    manifest = {"conf_min": ARCHIVE_CONF_MIN, "n_total": len(recs), "n_kept": len(kept),
                "cases": sorted(recs, key=lambda r: -r["conf_fg"])}
    with open(join(OUT_DIR, "archive_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=1)
    confs = sorted(r["conf_fg"] for r in recs)
    print(f"[predict] кейсов: {len(recs)}, прошло фильтр: {len(kept)}")
    print(f"[predict] conf_fg: min {confs[0]:.3f} | медиана {confs[len(confs)//2]:.3f} | max {confs[-1]:.3f}")
    if confs[0] < 0.5:
        print("[predict] ВНИМАНИЕ: есть кейсы с conf < 0.5 — проверьте их визуально перед train")
    print(f"[predict] манифест: {join(OUT_DIR, 'archive_manifest.json')}")

In [ ]:
# ЯЧЕЙКА 9 — точка остановки predict-фазы
# ============================================================
if not RUN_TRAIN:
    print("=" * 70)
    print("[PHASE=predict] ГОТОВО. Дальше ВРУЧНУЮ:")
    print("  1) Save Version → скачайте output/archive_out/")
    print("     (*.b2nd + *_seg.b2nd + *.pkl + *_conf.json + archive_manifest.json)")
    print("  2) Залейте как приватный датасет iliaszalialov/distill-archive-pseudo")
    print("     (веб-интерфейс; API-заливка с KGAT отдаёт 500)")
    print("  3) Новая версия ноутбука: PHASE=train, приаттачить distill-archive-pseudo")
    print("=" * 70)
else:
    print("[run] PHASE=train — продолжаем")


In [ ]:
# ЯЧЕЙКА 10 — [PHASE=train] GT-сплит 85/15 (seed 20261003, как в v7) + merge архива
# Архивные кейсы добавляются ТОЛЬКО в train (val — чистый GT).
# ============================================================
if not RUN_TRAIN:
    print("[train] PHASE=predict — ячейка пропущена")
else:
    real_dirs = [d for d in glob.glob(os.path.join(DATASET_DIR, "*_3d_fullres"))
                 if os.path.isdir(d) and os.listdir(d)]
    assert real_dirs, f"нет папки данных *_3d_fullres в {DATASET_DIR}"
    real_data_dir = real_dirs[0]
    expected_data_dir = os.path.join(DATASET_DIR, "nnUNetResEncUNetMPlans_3d_fullres")
    if os.path.realpath(expected_data_dir) != os.path.realpath(real_data_dir) \
            and not os.path.exists(expected_data_dir):
        os.symlink(real_data_dir, expected_data_dir)
    DATA_DIR = real_data_dir
    print(f"[setup] папка данных: {DATA_DIR}")

    cases = sorted(os.path.splitext(f)[0] for f in os.listdir(DATA_DIR)
                   if f.endswith(".b2nd") and not f.endswith("_seg.b2nd")
                   and not f.startswith("arch_"))
    assert cases, f"нет GT-данных в {DATA_DIR}"
    rng = np.random.RandomState(20261003)                 # ⚙ тот же сид, что в v7
    perm = rng.permutation(len(cases))
    n_val = max(4, int(round(len(cases) * 0.15)))
    val_ids = [cases[i] for i in perm[:n_val]]
    train_ids = [cases[i] for i in perm[n_val:]]
    print(f"[setup] GT-сплит: train {len(train_ids)}, val {len(val_ids)}")

    # ---- merge архива ----
    arch_dirs = sorted(glob.glob("/kaggle/input/*archive*"))
    arch_dirs = [d for d in arch_dirs if os.path.isdir(join(d, "archive_out"))]
    assert arch_dirs, "Не найден датасет с псевдо-масками (distill-archive-pseudo) — приаттачьте его"
    ARCH_PP = arch_dirs[0]
    ARCH_OUT = join(ARCH_PP, "archive_out")
    with open(join(ARCH_OUT, "archive_manifest.json")) as f:
        manifest = json.load(f)
    kept = sorted((c for c in manifest["cases"] if c.get("kept")), key=lambda c: -c["conf_fg"])
    sel = kept[:N_ARCHIVE]
    assert sel, "архив: 0 кейсов прошло фильтр уверенности — снизьте ARCHIVE_CONF_MIN и перезапустите predict"
    if SMOKE:
        sel = sel[:2]
    print(f"[archive] в манифесте {manifest['n_total']} кейсов, прошло фильтр {manifest['n_kept']}, "
          f"берём топ-{len(sel)} (conf {sel[0]['conf_fg']:.3f}..{sel[-1]['conf_fg']:.3f})")
    n_linked = 0
    for c in sel:
        for suf in (".b2nd", "_seg.b2nd", ".pkl"):
            src = join(ARCH_OUT, c["case"] + suf)
            assert os.path.isfile(src), f"нет файла {src} — датасет псевдо-масок неполный"
            dst = join(DATA_DIR, c["case"] + suf)
            if not os.path.exists(dst):
                try:
                    os.symlink(src, dst)
                except OSError:
                    shutil.copy2(src, dst)
                n_linked += 1
    train_ids = train_ids + [c["case"] for c in sel]
    if SMOKE:
        train_ids = train_ids[:8]
        val_ids = val_ids[:3]
    SPLITS_PATH = os.path.join(DATASET_DIR, "splits_final.json")
    with open(SPLITS_PATH, "w") as f:
        json.dump([{"train": train_ids, "val": val_ids}], f)
    print(f"[setup] ИТОГО train {len(train_ids)} (GT {len(train_ids) - len(sel)} + архив {len(sel)}), "
          f"val {len(val_ids)} (только GT); линковано файлов: {n_linked}")

In [ ]:
# ЯЧЕЙКА 11 — софт-таргеты учителя и функции лосса
# Итоговая формула (на каждом уровне i deep supervision, wᵢ = 1/2ⁱ):
#   L = Σᵢ wᵢ · [ W_DICE·softDice(sᵢ, p̄ₜ) + (1−W_DICE)·softCE(sᵢ, p̄ₜ) ]
# hard-GT (HARD_GT_WEIGHT>0): L = (1−w)·soft + w·hard; на GT-кейсах якорь
# истинных масок, на архивных — уверенный self-training по псевдо-маскам.
# ============================================================
@torch.no_grad()
def teacher_probs(data):
    acc = None
    d = data.to(teacher_device, non_blocking=True)
    if not CPU_MODE:
        d = d.half()
    for net in teacher_nets:
        logits = net(d)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
        p = torch.softmax(logits.float(), dim=1)
        acc = p if acc is None else acc + p
    return (acc / len(teacher_nets)).detach().to(data.device)

def soft_targets_for_ds(t_probs, out_shapes):
    outs = []
    for shp in out_shapes:
        if tuple(t_probs.shape[2:]) == tuple(shp):
            outs.append(t_probs)
        else:
            outs.append(torch.nn.functional.interpolate(
                t_probs, size=shp, mode="area").clamp_min(1e-6))
    return outs

def soft_ce(s_logits, t_probs):
    logp = torch.log_softmax(s_logits.float(), dim=1)
    return -(t_probs * logp).sum(dim=1).mean()

def soft_dice(s_logits, t_probs, eps=1e-6):
    s = torch.softmax(s_logits.float(), dim=1)
    dims = (0, 2, 3, 4)
    inter = (s * t_probs).sum(dims)
    denom = s.sum(dims) + t_probs.sum(dims)
    return (1.0 - (2.0 * inter + eps) / (denom + eps)).mean()

In [ ]:
# ЯЧЕЙКА 12 — страховочный аплоад чекпоинта (СВОЙ датасет цепочки архива!)
# slug = nnunet-archive-ckpt — НЕ nnunet-distill-ckpt (там чекпоинты v7,
# их приаттачивать сюда нельзя: resume-гвард возьмёт чужую эпоху).
# ============================================================
class _SessionOver(Exception):
    # Мягкое завершение по бюджету: чекпоинт уже сохранён.
    pass

def kaggle_ckpt_upload(path, notes):
    try:
        import requests
        tok = os.environ.get("KAGGLE_BEARER")
        if not tok:
            print("[ckpt-upload] нет KAGGLE_BEARER — пропуск")
            return
        H = {"Authorization": f"Bearer {tok}"}
        base = "https://www.kaggle.com/api/v1"
        fname = os.path.basename(path)
        size = os.path.getsize(path)
        mtime = int(os.path.getmtime(path) * 1000)   # МИЛЛИСЕКУНДЫ (API Kaggle ожидает мс)
        r = requests.post(f"{base}/datasets/upload/file/{size}/{mtime}",
                          params={"fileName": fname}, headers=H, timeout=300)
        r.raise_for_status()
        info = r.json()
        with open(path, "rb") as f:
            pr = requests.put(info["createUrl"], data=f, timeout=900)  # НЕ 7200
        pr.raise_for_status()
        owner, slug = "iliaszalialov", "nnunet-archive-ckpt"
        vbody = {"versionNotes": notes, "files": [{"token": info["token"]}], "convertToCsv": False}
        r = requests.post(f"{base}/datasets/create/version/{owner}/{slug}",
                          json=vbody, headers=H, timeout=300)
        if r.status_code == 404:
            cbody = {"title": "nnunet archive ckpt", "slug": slug, "ownerSlug": owner,
                     "licenseName": "CC0-1.0", "isPrivate": True,
                     "files": vbody["files"], "convertToCsv": False}
            r = requests.post(f"{base}/datasets/create/new", json=cbody, headers=H, timeout=300)
        r.raise_for_status()
        print(f"[ckpt-upload] ок ({notes})")
    except Exception as e:
        body = getattr(getattr(e, "response", None), "text", "")
        print(f"[ckpt-upload] ОШИБКА (обучение продолжается): {e} {body[:300]}")

In [ ]:
# ЯЧЕЙКА 13 — трейнер дистилляции (+ DA5 через MRO)
# Класс называется nnUNetTrainerDistill — та же папка output/resume-глоб, что в v7.
# ============================================================
_DA5 = None
if USE_DA5:
    try:
        from nnunetv2.training.nnUNetTrainer.variants.data_augmentation.nnUNetTrainerDA5 import (
            nnUNetTrainerDA5 as _DA5)
        print("[DA5] nnUNetTrainerDA5 найден — сильная аугментация активна")
    except ImportError:
        print("[DA5] нет в этой версии nnunetv2 — работаю без DA5")

_BASE = _DA5 if _DA5 is not None else nnUNetTrainerNoMirroring

class nnUNetTrainerDistill(_BASE):
    def __init__(self, plans, configuration, fold, dataset_json, device="cuda:0"):
        super().__init__(plans, configuration, fold, dataset_json, device=torch.device(device))
        self.num_epochs = 1 if SMOKE else NUM_EPOCHS
        if SMOKE:
            self.num_iterations_per_epoch = 30
            self.num_val_iterations_per_epoch = 5
        else:
            self.num_val_iterations_per_epoch = 10
        self.initial_lr = PEAK_LR
        self.oversample_foreground_percent = OVERSAMPLE
        self.save_every = 5
        self._t0 = None
        self._iter_times = []

    def setup_DA_params(self):
        super().setup_DA_params()
        dap = getattr(self.configuration_manager, "data_aug_params", None)
        if isinstance(dap, dict):
            dap["mirror_axes"] = None                 # студент без mirroring (в т.ч. под DA5)
            self.print_to_log_file("[DA] mirror_axes -> None")

    def configure_rotation_dummyDA_mirroring_and_inital_DA_scales(self):
        res = super().configure_rotation_dummyDA_mirroring_and_inital_DA_scales()
        # сигнатура менялась между версиями: 3- или 4-tuple (mirror_axes последняя)
        if isinstance(res, tuple) and len(res) in (3, 4):
            self.print_to_log_file(f"[DA] mirror_axes {res[-1]} -> None")
            return res[:-1] + (None,)
        return res

    def on_train_start(self):
        self._t0 = time.time()
        super().on_train_start()
        if self.lr_scheduler is not None:
            self.lr_scheduler.initial_lr = PEAK_LR
            if hasattr(self.lr_scheduler, "max_steps"):
                self.lr_scheduler.max_steps = self.num_epochs
            self.print_to_log_file(f"[lr] peak LR = {PEAK_LR}")
        dap = getattr(self.configuration_manager, "data_aug_params", {})
        self.print_to_log_file(f"[DA] effective mirror_axes: {dap.get('mirror_axes', 'n/a')}")
        # тёплый старт: foldN учителя (fold1 выбран 2026-11-03) или готовый дистиллят
        if INIT_SOURCE != "none" and self.current_epoch == 0 and self.local_rank == 0:
            if INIT_SOURCE.startswith("fold"):
                _f = int(INIT_SOURCE[4:])
                _src = join(IN_RESULTS, f"fold_{_f}", "checkpoint_final.pth")
            elif INIT_SOURCE == "distill":
                _src = find_one("/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_final.pth")
            else:
                _src = None
            assert _src and os.path.isfile(_src), f"[init] INIT_SOURCE={INIT_SOURCE}: источник весов не найден"
            sd = torch.load(_src, map_location="cpu", weights_only=False)
            net = self.network
            if hasattr(net, "_orig_mod"):
                net = net._orig_mod
            net.load_state_dict(sd["network_weights"])  # strict
            print(f"[init] тёплый старт студента из {_src}: OK (strict)")
            del sd
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        # resume: долив нейтральных записей логгера (страховка от IndexError в EMA)
        if self.current_epoch > 0:
            L = self.logger.my_fantastic_logging
            n = self.current_epoch - len(L["ema_fg_dice"])
            if n > 0:
                print(f"[logger] resume: доливаю {n} записей")
                L["mean_fg_dice"].extend([0.0] * n)
                L["train_losses"].extend([0.0] * n)
                L["val_losses"].extend([0.0] * n)
                L["lrs"].extend([PEAK_LR] * n)
                L["epoch_start_timestamps"].extend([0] * n)
                L["epoch_end_timestamps"].extend([0] * n)
                L["dice_per_class_or_region"].extend([None] * n)
                v = 0.0
                for _ in range(n):
                    v = v * 0.9 + 0.1 * 0.9
                    L["ema_fg_dice"].append(v)

    def on_epoch_end(self):
        super().on_epoch_end()
        if self.local_rank == 0 and (self.current_epoch + 1) % UPLOAD_EVERY == 0:
            kaggle_ckpt_upload(join(self.output_folder, "checkpoint_latest.pth"),
                               f"epoch {self.current_epoch}")
        if self._t0 is not None and (time.time() - self._t0) > BUDGET_H * 3600:
            print(f"[budget] {BUDGET_H} ч исчерпаны — чисто завершаю на эпохе {self.current_epoch}")
            self.save_checkpoint(join(self.output_folder, "checkpoint_latest.pth"))
            kaggle_ckpt_upload(join(self.output_folder, "checkpoint_latest.pth"),
                               f"epoch {self.current_epoch} (budget stop)")
            raise _SessionOver

    def train_step(self, batch: dict) -> dict:
        if SMOKE and len(self._iter_times) < 5:
            t_iter = time.time()
        else:
            t_iter = None
        data = batch["data"]
        target = batch["target"]
        data = data.to(self.device, non_blocking=True)
        if isinstance(target, list):
            target = [i.to(self.device, non_blocking=True) for i in target]
        else:
            target = target.to(self.device, non_blocking=True)
        self.optimizer.zero_grad(set_to_none=True)
        with torch.autocast(self.device.type, enabled=self.device.type == "cuda" and not CPU_MODE):
            output = self.network(data)
            shapes = [tuple(o.shape[2:]) for o in output]
            with torch.no_grad():
                tp = teacher_probs(data)
            t_list = soft_targets_for_ds(tp, shapes)
            n = len(output)
            ws = [1.0 / (2 ** i) for i in range(n)]   # веса DS — менять не рекомендую
            ws = [w / sum(ws) for w in ws]
            l_soft = sum(w * (W_DICE * soft_dice(o, t) + (1 - W_DICE) * soft_ce(o, t))
                         for w, o, t in zip(ws, output, t_list))
            l = l_soft
            if HARD_GT_WEIGHT > 0:
                l = (1 - HARD_GT_WEIGHT) * l_soft + HARD_GT_WEIGHT * self.loss(output, target)
        if self.grad_scaler is not None:
            self.grad_scaler.scale(l).backward()
            self.grad_scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.grad_scaler.step(self.optimizer)
            self.grad_scaler.update()
        else:
            l.backward()
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.optimizer.step()
        if t_iter is not None:
            dt = time.time() - t_iter
            self._iter_times.append(dt)
            vram = f"{torch.cuda.memory_allocated() / 1e9:.2f} ГБ" if torch.cuda.is_available() else "cpu"
            print(f"[smoke] итерация {len(self._iter_times)}: {dt:.2f} с (VRAM {vram})")
        return {"loss": l.detach().cpu().numpy()}

In [ ]:
# ЯЧЕЙКА 14 — resume: поиск чекпоинта, защита от чужих чекпоинтов
# СВОЯ цепочка = nnunet-archive-ckpt (плоский checkpoint_latest).
# Чекпоинты v7 (nnunet-distill-ckpt) НЕ приаттачивать: гвард остановит, но лучше не подсовывать.
# ============================================================
prev_ckpt = (find_one("/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_latest.pth")
             or next((p for p in sorted(glob.glob("/kaggle/input/**/checkpoint_latest.pth", recursive=True))
                      if "nnUNetTrainerNoMirroring" not in p), None))
if prev_ckpt:
    print(f"[resume] найден чекпоинт прошлой сессии: {prev_ckpt}")
    _sd = torch.load(prev_ckpt, map_location="cpu", weights_only=False)
    print(f"[resume]   current_epoch={_sd.get('current_epoch')}, best_ema={_sd.get('_best_ema')}")
    del _sd
elif REQUIRE_CKPT and INIT_SOURCE == "none":
    sys.exit("[resume] чекпоинт не найден, REQUIRE_CKPT=1 и INIT=none — СТОП, не жгу квоту")
elif REQUIRE_CKPT:
    print("[resume] своего чекпоинта нет — старт через тёплые веса INIT="
          f"{INIT_SOURCE} (первая сессия цепочки)")

plans_trainer = dict(plans)
plans_trainer.setdefault("continue_training", prev_ckpt is not None)
trainer = nnUNetTrainerDistill(plans_trainer, CONFIG, 0, dataset_json,
                               device="cpu" if CPU_MODE else "cuda:0")
RES_DIR = trainer.output_folder
os.makedirs(RES_DIR, exist_ok=True)
if prev_ckpt:
    shutil.copy2(prev_ckpt, join(RES_DIR, "checkpoint_latest.pth"))
    kaggle_ckpt_upload(join(RES_DIR, "checkpoint_latest.pth"),
                       "тест аплоада при старте")

ckpt_latest = join(RES_DIR, "checkpoint_latest.pth")
need_training = True
if os.path.isfile(ckpt_latest):
    trainer.load_checkpoint(ckpt_latest)
    print(f"[resume] продолжаем с эпохи {trainer.current_epoch}")
    if not SMOKE and trainer.current_epoch >= NUM_EPOCHS:
        need_training = False
        print(f"[run] эпоха {trainer.current_epoch} >= NUM_EPOCHS={NUM_EPOCHS} — "
              f"цикл завершён или чекпоинт чужой. Обучение пропущено.")
else:
    trainer.initialize()

In [ ]:
# ЯЧЕЙКА 15 — обучение
# ============================================================
if RUN_TRAIN:
    if need_training:
        try:
            trainer.run_training()
        except _SessionOver:
            print("[run] сессия завершена по бюджету — чекпоинт сохранён")
    trainer.save_checkpoint(ckpt_latest)
    kaggle_ckpt_upload(ckpt_latest, f"конец сессии, эпоха {trainer.current_epoch}")
    print("[run] ГОТОВО. Файлы:")
    for f in glob.glob(join(RES_DIR, "*.pth")):
        print("  ", f, f"{os.path.getsize(f) / 1e6:.0f} МБ")
else:
    print("[train] PHASE=predict — обучение пропущено")

In [ ]:
# ЯЧЕЙКА 16 — финализация ТОЛЬКО когда цикл действительно завершён
# ============================================================
if RUN_TRAIN:
    if trainer.current_epoch >= NUM_EPOCHS and not SMOKE:
        shutil.copy2(ckpt_latest, join(RES_DIR, "checkpoint_final.pth"))
        print(f"[final] цикл завершён на эпохе {trainer.current_epoch}: checkpoint_final.pth сохранён")
        print("[final] честные числа — только внешний холдаут (локальная валидация 10 кейсов); "
              "pseudo-dice из лога НЕ сравним с полной валидацией")
    else:
        print(f"[final] цикл не завершён (эпоха {trainer.current_epoch}/{NUM_EPOCHS}) — "
              f"продолжайте следующей сессией")

## Этап B: квантизация INT8 (NNCF) — только после завершения цикла

Восстанавливаем студента из `checkpoint_final.pth`, экспортируем ONNX →
OpenVINO IR FP32 → NNCF INT8 PTQ (калибровка на патчах валидационных томов).
Результат: `student_fp32.xml/bin` и `student_int8.xml/bin` в output сессии.

In [ ]:
# ЯЧЕЙКА 17 — экспорт IR + INT8-квантизация студента (если цикл ЗАВЕРШЁН)
# nncf 3.4 + openvino latest (nncf НЕ тянет openvino сам — extra)
# ============================================================
if not RUN_TRAIN:
    print("[quant] PHASE=predict — квантизация пропущена")
else:
    CYCLE_DONE = (trainer.current_epoch >= NUM_EPOCHS) and not SMOKE
    if not CYCLE_DONE:
        print(f"[quant] цикл не завершён ({trainer.current_epoch}/{NUM_EPOCHS}) — квантизация пропущена")
    else:
        try:
            import nncf
            import openvino  # noqa: F401
            print(f"[quant] nncf {nncf.__version__}")
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "openvino", "nncf==3.4.0", "onnxscript"], check=True)
            import nncf
            import openvino  # noqa: F401

        import openvino as ov
        import blosc2

        net_q = build_net()
        sd_q = torch.load(join(RES_DIR, "checkpoint_final.pth"), map_location="cpu", weights_only=False)
        net_q.load_state_dict(sd_q["network_weights"])
        net_q.eval()

        dummy = torch.randn(1, 1, 56, 192, 192)
        onnx_path = join(RES_DIR, "student_model.onnx")
        export_kwargs = dict(input_names=["data"], output_names=["logits"],
                             dynamic_axes={"data": {0: "batch"}, "logits": {0: "batch"}},
                             opset_version=11)
        try:
            torch.onnx.export(net_q, dummy, onnx_path, dynamo=False, **export_kwargs)
        except TypeError:  # старый torch без параметра dynamo
            torch.onnx.export(net_q, dummy, onnx_path, **export_kwargs)
        print(f"[quant] ONNX: {onnx_path}")

        ov_model = ov.convert_model(onnx_path)
        ir_fp32 = join(RES_DIR, "student_fp32.xml")
        ov.serialize(ov_model, ir_fp32)
        print(f"[quant] IR FP32: {ir_fp32}")

        # калибровка: патчи 56x192x192 из ВАЛИДАЦИОННЫХ GT-томов (b2nd, уже
        # нормализованы; архивные кейсы в val не попадают — сплит их туда не кладёт)
        calib = []
        rng_c = np.random.RandomState(20261003)
        val_files = [join(DATA_DIR, c + ".b2nd") for c in val_ids][:6]
        for vf in val_files:
            vol = np.asarray(blosc2.open(vf))
            vol = vol.squeeze(0) if vol.ndim == 4 else vol
            for _ in range(4):
                z0 = rng_c.randint(0, max(1, vol.shape[0] - 56))
                y0 = rng_c.randint(0, max(1, vol.shape[1] - 192))
                x0 = rng_c.randint(0, max(1, vol.shape[2] - 192))
                patch = vol[z0:z0 + 56, y0:y0 + 192, x0:x0 + 192].astype(np.float32)
                if patch.shape == (56, 192, 192):
                    calib.append(patch[None, None])
        print(f"[quant] калибровка: {len(calib)} патчей")

        # nncf 3.x: Dataset(список готовых патчей) без transform_fn; 2.x — fallback
        try:
            from nncf.data import Dataset as NncfDataset
            nncf_ds = NncfDataset(calib)
            quant_model = nncf.quantize(ov_model, nncf_ds,
                                        preset=nncf.QuantizationPreset.MIXED,
                                        subset_size=len(calib))
        except TypeError:
            quant_model = nncf.quantize(ov_model,
                                        calibration_data_fn=lambda: iter(calib),
                                        preset=nncf.QuantizationPreset.MIXED,
                                        subset_size=len(calib))
        ir_int8 = join(RES_DIR, "student_int8.xml")
        ov.serialize(quant_model, ir_int8)
        print(f"[quant] IR INT8: {ir_int8}")
        print("[quant] ГОТОВО: student_fp32.xml/bin + student_int8.xml/bin в output")

## Этап C: валидация INT8-студента на val-сплите (Kaggle-шкала)

Sliding-window инференс INT8-модели на val-кейсах + Dice по органам.
⚠ Эти числа — **внутренняя (Kaggle) шкала**: сплит контаминирован учителями
(они обучались на этих же кейсах), Dice завышён. Прод-гейт считается
**локально на внешнем холдауте** (10 кейсов, не виденных учителями):
mean ≥ 0.8394 и малые структуры ≥ −0.03.

In [ ]:
# ЯЧЕЙКА 18 — sliding-window валидация INT8 + Dice по органам
# Упрощённый движок (равномерные веса, без crop тела): для контроля достаточно
# ============================================================
if not RUN_TRAIN:
    print("[val] PHASE=predict — валидация пропущена")
else:
    CYCLE_DONE = (trainer.current_epoch >= NUM_EPOCHS) and not SMOKE
    if not CYCLE_DONE:
        print("[val] цикл не завершён — валидация пропущена")
    else:
        try:
            import openvino.runtime as ovrt
        except ModuleNotFoundError:
            import openvino as ovrt
        import blosc2

        core_v = ovrt.Core()
        comp_v = core_v.compile_model(ir_int8, "CPU")
        req_v = comp_v.create_infer_request()
        in_name = comp_v.input(0).any_name
        out_name = comp_v.output(0).any_name

        PZ, PY, PX = 56, 192, 192
        STRIDE = 0.5

        def infer_volume(vol):
            steps = []
            for d, p in zip(vol.shape, (PZ, PY, PX)):
                if d < p:
                    steps.append([0])
                else:
                    s = max(1, int(p * STRIDE))
                    st = list(range(0, d - p + 1, s))
                    if not st or st[-1] + p < d:
                        st.append(d - p)
                    steps.append(st)
            logits = np.zeros((32,) + vol.shape, dtype=np.float32)
            wsum = np.zeros(vol.shape, dtype=np.float32)
            for z0 in steps[0]:
                for y0 in steps[1]:
                    for x0 in steps[2]:
                        patch = vol[z0:z0 + PZ, y0:y0 + PY, x0:x0 + PX]
                        res = req_v.infer({in_name: patch[None, None]})
                        out = res[out_name]
                        logits[:, z0:z0 + PZ, y0:y0 + PY, x0:x0 + PX] += out[0]
                        wsum[z0:z0 + PZ, y0:y0 + PY, x0:x0 + PX] += 1
            logits /= np.maximum(wsum, 1e-8)
            return np.argmax(logits, axis=0).astype(np.uint8)

        def dice(a, b):
            a, b = a.astype(bool), b.astype(bool)
            denom = int(a.sum()) + int(b.sum())
            if denom == 0:
                return None
            return 2.0 * (a & b).sum() / denom

        for case in val_ids[:5]:
            vf = join(DATA_DIR, case + ".b2nd")
            sf = join(DATA_DIR, case + "_seg.b2nd")
            vol = np.asarray(blosc2.open(vf))
            vol = vol.squeeze(0) if vol.ndim == 4 else vol
            gt = np.asarray(blosc2.open(sf))
            gt = gt.squeeze(0) if gt.ndim == 4 else gt
            seg = infer_volume(vol.astype(np.float32))
            dices = {}
            for name, lbl in dataset_json["labels"].items():
                if lbl == 0:
                    continue
                d = dice(seg == lbl, gt == lbl)
                if d is not None:
                    dices[name] = float(d)
            m = np.mean(list(dices.values()))
            print(f"[val] {case}: mean Dice = {m:.4f} ({len(dices)} органов)")
        print("[val] напоминание: сплит контаминирован учителями — честные числа "
              "только на внешнем холдауте (локально)")